# 00. Configurazione e prima decisione

Questo notebook e il punto di partenza. Configuriamo l'ambiente, capiamo come viene scelto un backend e facciamo una prima chiamata di decisione.

Un decision model e diverso da un chatbot. Non scrive una frase. Legge uno stato (le informazioni che può vedere) e un insieme di domande le cui risposte possibili sono dichiarate in anticipo. Restituisce poi una probabilità per ogni risposta dichiarata. Questa piccola differenza cambia il modo di costruire i sistemi: le probabilità si possono sogliare, confrontare e verificare.

Tutto in questo repository e gratuito o open. Partiamo con un mock offline così il notebook funziona su qualunque macchina senza chiavi, download o GPU.


![Flusso di un decision model](../images/decision_model_flow.png)

*Il modello legge uno stato e restituisce una probabilità per ogni risposta dichiarata. Non scrive mai una frase.*


## Passo 1. Verificare l'interprete Python

Il progetto fissa Python 3.12 nel file `.python-version`. `uv` crea un ambiente virtuale isolato in `.venv`, quindi non viene installato nulla nel Python di sistema. La cella qui sotto riporta solo la versione dell'interprete.


In [ ]:
import sys
print(sys.version.split()[0])


## Passo 2. Importare il pacchetto del progetto

Il pacchetto e installato in modalita editabile da `uv sync`. questo significa che `decision_qa` punta alla cartella locale `src/`: ogni modifica al codice e subito visibile, senza reinstallazione. Stampare il percorso conferma che stiamo importando il nostro codice.


In [ ]:
import decision_qa
print(decision_qa.__file__)


## Passo 3. Caricare la configurazione

`load_config()` legge il file `.env` (se presente) e restituisce una piccola dataclass. Sono supportati due backend: `local` per modelli open che girano sulla propria macchina, e `hosted` per una route con piano gratuito. Il predefinito e `local`, quindi non serve alcun segreto per seguire i notebook.


In [ ]:
from decision_qa import load_config
cfg = load_config()
print(cfg.backend)


La dataclass contiene anche il nome del modello e l'URL di base. Li leggiamo separatamente cosi ogni cella mostra esattamente una idea.


In [ ]:
print(cfg.model)
print(cfg.base_url)


## Passo 4. Costruire l'URL dell'endpoint

L'API che usiamo si chiama `/v1/systemone`. Alcuni URL di base finiscono gia con `/v1`, quindi la proprieta `endpoint` rimuove il suffisso duplicato. E un dettaglio piccolo, ma un URL sbagliato e il primo errore piu comune. L'assert verifica il risultato.


In [ ]:
print(cfg.endpoint)
assert cfg.endpoint.endswith('/v1/systemone')
print('endpoint OK')


## Passo 5. Scegliere un client

Un client e l'oggetto che invia una richiesta e restituisce un risultato. `MockClient` e deterministico, gratuito e offline. Serve per imparare le strutture dati e per testare il codice. `SystemOneClient` parla con un server reale quando disponibile. Creiamo entrambi, ma per ora chiamiamo solo il mock.


In [ ]:
from decision_qa import MockClient
mock = MockClient(seed=42)
print(type(mock).__name__)


In [ ]:
from decision_qa import SystemOneClient
live = SystemOneClient(cfg)
print(type(live).__name__)


## Passo 6. Definire uno stato

Lo stato e cio che il modello puo guardare. In un task di question answering contiene di solito un brano di testo e una domanda. Tenerlo come dizionario lo rende facile da estendere in seguito con campi di tipo JSON.


In [ ]:
state = {
    'context': 'A trial of 240 patients reports a 42 percent reduction in headaches.',
    'question': 'Is the treatment effective?',
}
print(state['question'])


In [ ]:
assert 'context' in state and 'question' in state
print('state keys OK')


## Passo 7. La domanda Noul

Noul chiede se un'affermazione e vera. E il paradigma piu semplice: le risposte dichiarate sono `true` e `false`. Aggiungiamo un campo `gold` solo per didattica e valutazione. Una richiesta reale non lo includerebbe.


In [ ]:
from decision_qa import NoulQuestion
noul = NoulQuestion(label='supported', statement='The treatment is effective.', gold=True)
print(noul.type, noul.label)


In [ ]:
from decision_qa import question_labels
print(question_labels(noul))
assert question_labels(noul) == ['true', 'false']


## Passo 8. La domanda Choice

Choice chiede al modello di scegliere tra piu opzioni dichiarate. Le opzioni sono fissate da noi, non generate dal modello. `gold_index` indica l'opzione corretta e serve solo per il punteggio.


In [ ]:
from decision_qa import ChoiceQuestion
choice = ChoiceQuestion(label='verdict', options=['effective', 'ineffective', 'unclear'], gold_index=0)
print(choice.options)


In [ ]:
print(question_labels(choice))
assert 0 <= choice.gold_index < len(choice.options)
print('choice OK')


## Passo 9. La domanda Score

Score usa una scala ordinata. A differenza di Choice, l'ordine ha significato: `strong` e meglio di `moderate`. Non bisogna mai mescolare i livelli di una domanda Score.


In [ ]:
from decision_qa import ScoreQuestion
score = ScoreQuestion(label='evidence', rubric='Rate the strength of evidence.', levels=['weak', 'moderate', 'strong'], gold_index=2)
print(score.levels)


## Passo 10. Costruire una richiesta

I tre tipi di domanda vengono tradotti in un unico payload che rispecchia il formato della richiesta `/v1/systemone`. Il `label` non fa parte del formato di rete: e il nostro identificativo per rileggere i risultati.


In [ ]:
from decision_qa import to_systemone_payload
import json
payload = to_systemone_payload(state, [noul, choice, score])
print(json.dumps(payload, indent=2))


In [ ]:
assert len(payload['questions']) == 3
print('three questions in one payload')


## Passo 11. Fare la prima chiamata

Chiamiamo il client con lo stato e una sola domanda Noul. L'oggetto risultato contiene l'etichetta della domanda e un dizionario di probabilita. Lo esaminiamo un pezzo alla volta.


In [ ]:
results = mock.decision(state, [noul])
print(results[0].question_label)


In [ ]:
probs = results[0].probabilities
print(probs)


In [ ]:
assert abs(sum(probs.values()) - 1.0) < 1e-9
print('probabilities sum to one')


## Passo 12. Fare tutte e tre le domande insieme

Il batch e uno dei principali vantaggi di questo formato. Uno stato e piu domande viaggiano in una sola chiamata, piu economica e veloce di tre chiamate separate.


In [ ]:
all_results = mock.decision(state, [noul, choice, score])
print([r.question_label for r in all_results])


In [ ]:
assert len(all_results) == 3
print('one result per question')


In [ ]:
for r in all_results:
    print(r.question_label, max(r.probabilities, key=r.probabilities.get))


Un'immagine rende gli stessi dati piu facili da leggere. I tre pannelli qui sotto mostrano una distribuzione per domanda. Nota che ogni pannello somma a uno.


In [ ]:
import matplotlib.pyplot as plt
palette = ['#4C78A8', '#F58518', '#54A24B']
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, result, color in zip(axes, all_results, palette):
    labels = list(result.probabilities)
    values = list(result.probabilities.values())
    ax.bar(labels, values, color=color)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability')
    ax.set_title(result.question_label)
    ax.grid(axis='y', alpha=0.25)
    for index, value in enumerate(values):
        ax.text(index, value + 0.02, f'{value:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## Passo 13. Cambiare lo stato e osservare l'effetto

La stessa domanda puo ricevere una distribuzione diversa quando cambiano le evidenze. Questo e il comportamento fondamentale che sfrutteremo dopo: le decisioni dipendono dallo stato.


In [ ]:
weak_state = {'context': 'A five-patient anecdote mentions improvement.', 'question': state['question']}
weak_results = mock.decision(weak_state, [noul])
print(weak_results[0].probabilities)


In [ ]:
strong = results[0].probabilities['true']
weak = weak_results[0].probabilities['true']
print(round(strong, 3), round(weak, 3))
assert strong > 0 and weak > 0


## Riepilogo

Abbiamo configurato un backend, costruito i tre tipi di domanda, li abbiamo racchiusi in un unico payload e letto una distribuzione di probabilita. Il notebook successivo approfondisce ogni paradigma, incluse le soglie di confidenza e l'invarianza all'ordine delle opzioni.
